# Phần IV — Bài 3: Luật kết hợp thăng chức

Chỉ dùng **B = {Tuổi, Năng lực làm việc, Thăng chức}**, tìm các luật có **vế phải chỉ là thuộc tính Thăng chức**, **min_sup = 30%** và **min_conf = 80%**.

**Họ tên:** Nguyễn Lộc  \n**MSSV:** 6451071043  \n**Nguồn đề:** *Tàu liệu LAB1.pdf*, trang 43–45.  \n**Dữ liệu:** nhập trực tiếp từ bảng trong PDF; không cần tải thêm file.

> Chạy **Runtime → Run all** trên Colab. Mỗi notebook độc lập và có lệnh in họ tên/MSSV.

In [1]:
HO_TEN='NGUYỄN LỘC'
MSSV='6451071043'
print(f'Họ tên: {HO_TEN} | MSSV: {MSSV}')

Họ tên: NGUYỄN LỘC | MSSV: 6451071043


In [2]:
from itertools import combinations
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def support_count(transactions, itemset):
    s=set(itemset)
    return sum(s.issubset(set(t)) for t in transactions)

def apriori_manual(transactions, min_support_count):
    """Thuật toán Apriori: join + prune ở mỗi mức k."""
    transactions=[frozenset(t) for t in transactions]
    n=len(transactions)
    all_items=sorted(set().union(*transactions))
    candidates=[frozenset([x]) for x in all_items]
    levels=[]; counts={}
    k=1
    while candidates:
        level={c:support_count(transactions,c) for c in candidates}
        frequent={c:cnt for c,cnt in level.items() if cnt>=min_support_count}
        if not frequent: break
        levels.append(frequent)
        counts.update(frequent)
        prev=set(frequent)
        candidate_next=set()
        for a,b in combinations(sorted(prev,key=lambda z:tuple(sorted(z))),2):
            merged=a|b
            if len(merged)==k+1 and all(frozenset(s) in prev for s in combinations(merged,k)):
                candidate_next.add(merged)
        candidates=sorted(candidate_next,key=lambda x:tuple(sorted(x)))
        k+=1
    return levels,counts

def itemsets_table(counts,n):
    rows=[{'k':len(items),'Tập phổ biến':', '.join(sorted(items)), 'Support count':cnt, 'Support':round(cnt/n,5)} for items,cnt in counts.items()]
    return pd.DataFrame(rows).sort_values(['k','Tập phổ biến']).reset_index(drop=True) if rows else pd.DataFrame(columns=['k','Tập phổ biến','Support count','Support'])

def rules_from_counts(counts, transactions, min_conf, *, target_prefix=None):
    n=len(transactions)
    rows=[]
    for whole,both_count in counts.items():
        if len(whole)<2: continue
        for size in range(1,len(whole)):
            for left_tuple in combinations(sorted(whole),size):
                left=frozenset(left_tuple)
                right=whole-left
                if target_prefix is not None and not (len(right)==1 and next(iter(right)).startswith(target_prefix)):
                    continue
                left_count=support_count(transactions,left)
                right_count=support_count(transactions,right)
                conf=both_count/left_count
                if conf+1e-12>=min_conf:
                    rows.append({'Vế trái':', '.join(sorted(left)), 'Vế phải':', '.join(sorted(right)),
                                 'Support count':both_count,'Support':round(both_count/n,5),
                                 'Confidence':round(conf,5),'Lift':round(conf/(right_count/n),5)})
    return pd.DataFrame(rows).sort_values(['Confidence','Support','Vế trái'],ascending=[False,False,True]).reset_index(drop=True) if rows else pd.DataFrame(columns=['Vế trái','Vế phải','Support count','Support','Confidence','Lift'])

def plot_rules(rules,title):
    if len(rules)==0:
        print('Không có luật nào vượt qua ngưỡng nên không vẽ scatter.')
        return
    fig,ax=plt.subplots(figsize=(7,4.2))
    ax.scatter(rules['Support'],rules['Confidence'],s=65,alpha=.8)
    ax.set(xlabel='Support',ylabel='Confidence',title=title)
    ax.grid(alpha=.3)
    plt.tight_layout();plt.show()


## Bảng 10 nhân viên theo đề

In [3]:
# Bài 3 — 10 nhân viên; chỉ dùng B={Tuổi, Năng lực, Thăng chức}
raw=[
    ('20..25','Giỏi','Có'),
    ('20..25','Khá','Không'),
    ('26..30','Giỏi','Có'),
    ('31..40','T.Bình','Có'),
    ('26..30','T.Bình','Không'),
    ('26..30','Khá','Không'),
    ('31..40','Khá','Không'),
    ('26..30','Khá','Có'),
    ('>40','Giỏi','Không'),
    ('26..30','Giỏi','Có'),
]
employees=pd.DataFrame(raw,columns=['Tuổi','Năng lực','Thăng chức']);employees.index+=1
employees.index.name='STT'
display(employees)
transactions=[[f'Tuổi={t}',f'Năng lực={nl}',f'TC={tc}'] for t,nl,tc in raw]
min_count=3;min_conf=.80
print('Ngưỡng: support >= 3/10; confidence >= 80%. Vế phải chỉ gồm TC=Có hoặc TC=Không.')


,Tuổi,Năng lực,Thăng chức
STT,,,
1,20..25,Giỏi,Có
2,20..25,Khá,Không
3,26..30,Giỏi,Có
4,31..40,T.Bình,Có
5,26..30,T.Bình,Không
6,26..30,Khá,Không
7,31..40,Khá,Không
8,26..30,Khá,Có
9,>40,Giỏi,Không


Ngưỡng: support >= 3/10; confidence >= 80%. Vế phải chỉ gồm TC=Có hoặc TC=Không.


## Kiểm tra điều kiện vế phải và cả hai ngưỡng

In [4]:
levels,counts=apriori_manual(transactions,min_count)
print('Các tập phổ biến trong B:');display(itemsets_table(counts,len(transactions)))
rules=rules_from_counts(counts,transactions,min_conf,target_prefix='TC=')
print('Các luật có vế phải chỉ gồm TC, đạt cả 2 ngưỡng:',len(rules))
display(rules)
if rules.empty:
    print('KẾT LUẬN: Không có luật nào đồng thời đạt support >=30% và confidence >=80%.')
    print('Không được hạ ngưỡng hoặc báo luật dưới ngưỡng là hợp lệ.')
    print('Đối chiếu các luật tiềm năng (chưa đạt ngưỡng):')
    candidates=[]
    for whole,cnt in counts.items():
        for item in whole:
            if not item.startswith('TC=') or len(whole)<2:continue
            ant=whole-{item};ant_cnt=support_count(transactions,ant)
            candidates.append({'Vế trái':', '.join(sorted(ant)),'Vế phải':item,
                               'Support':cnt/len(transactions),'Confidence':cnt/ant_cnt})
    if candidates:
        display(pd.DataFrame(candidates).sort_values(['Confidence','Support'],ascending=False).head(10))


Các tập phổ biến trong B:


,k,Tập phổ biến,Support count,Support
0,1,Năng lực=Giỏi,4,0.4
1,1,Năng lực=Khá,4,0.4
2,1,TC=Có,5,0.5
3,1,TC=Không,5,0.5
4,1,Tuổi=26..30,5,0.5
5,2,"Năng lực=Giỏi, TC=Có",3,0.3
6,2,"Năng lực=Khá, TC=Không",3,0.3
7,2,"TC=Có, Tuổi=26..30",3,0.3


Các luật có vế phải chỉ gồm TC, đạt cả 2 ngưỡng: 0


,Vế trái,Vế phải,Support count,Support,Confidence,Lift


KẾT LUẬN: Không có luật nào đồng thời đạt support >=30% và confidence >=80%.
Không được hạ ngưỡng hoặc báo luật dưới ngưỡng là hợp lệ.
Đối chiếu các luật tiềm năng (chưa đạt ngưỡng):


,Vế trái,Vế phải,Support,Confidence
0,Năng lực=Giỏi,TC=Có,0.3,0.75
1,Năng lực=Khá,TC=Không,0.3,0.75
2,Tuổi=26..30,TC=Có,0.3,0.60
